In [1]:
import pandas as pd
import numpy as np
import ast

# Define options
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ['uniform', 'uneven']

# Initialize an empty list to store dataframes
results_list = []

# Loop through parameter combinations
for num_warehouses in num_warehouses_options:
    for num_customers in num_customers_options:
        for capacity_distribution in capacity_distribution_options:
            # Construct the filename
            filename = f'results/full_tables/full_results_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv'
            read_df = pd.read_csv(filename)

            # Keep the last stored result for each policy (in case of multiple runs)
            read_df = read_df.drop_duplicates(subset='Policy', keep='last').reset_index(drop=True)

            # Create a new dataframe
            new_df = pd.DataFrame()
            new_df['num_warehouses'] = [num_warehouses] * len(read_df)
            new_df['num_customers'] = [num_customers] * len(read_df)
            new_df['capacity_distribution'] = [capacity_distribution] * len(read_df)

            # Add policy columns
            new_df['policy'] = read_df['Policy']
            new_df['avg_reward'] = read_df['MeanReward']
            new_df['std_reward'] = read_df['StdReward']
            new_df['mean_time'] = read_df['MeanTime']
            new_df['std_time'] = read_df['StdTime']
            new_df['all_rewards'] = read_df['AllRewards'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_distance_ranks_n_vectors'] = read_df['AllFacilityProximity'].apply(lambda x: eval(x) if isinstance(x, str) else x)

            #Get All_Distance_ranks single vector
            new_df['all_distance_ranks_single_vector'] = new_df['all_distance_ranks_n_vectors'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])

            #Get All_Times single vector
            new_df['all_times_n_vectors'] = read_df['AllTimes'].apply(lambda x: eval(x) if isinstance(x, str) else x)
            new_df['all_times_single_vector'] = new_df['all_times_n_vectors'].apply(lambda x: [item for sublist in x for item in sublist] if isinstance(x, list) else [])

            # Append to list
            results_list.append(new_df)

# Concatenate all dataframes into a single DataFrame
results = pd.concat(results_list, ignore_index=True)

family_cols = ['num_warehouses', 'num_customers', 'capacity_distribution']


def _as_reward_array(x):
    """AllRewards: flat list of per-episode total rewards -> 1-D float array."""
    if isinstance(x, str):
        return np.asarray(ast.literal_eval(x), dtype=float)
    return np.asarray(x, dtype=float)


def _family_of(r):
    return results[(results['num_warehouses'] == r['num_warehouses']) &
                   (results['num_customers'] == r['num_customers']) &
                   (results['capacity_distribution'] == r['capacity_distribution'])]


# --------------------------------------------------------------------- PHS comparison
def _row_stats_phs(r):
    fam = _family_of(r)
    phs_cost = -_as_reward_array(fam[fam['policy'] == 'perfect_hindsight']['all_rewards'].iloc[0])
    cost_pi = -_as_reward_array(r['all_rewards'])

    assert len(cost_pi) == len(phs_cost), "episodes not aligned - cannot pair"
    assert (r['policy'] == 'perfect_hindsight') or ((cost_pi - phs_cost) >= 0).all(), \
        f"PHS not per-episode optimal vs {r['policy']} - episodes misaligned"

    delta = cost_pi - phs_cost
    avg_phs = phs_cost.mean()
    avg_delta, std_delta = delta.mean(), delta.std(ddof=1)
    return pd.Series({
        'delta_phs_vector': delta,
        'avg_delta_phs': avg_delta,
        'std_delta_phs': std_delta,
        'gap_phs_vector': delta / phs_cost * 100.0,
        'avg_gap_phs': avg_delta / avg_phs * 100.0,
        'std_gap_phs': std_delta / avg_phs * 100.0,
    })

results[['delta_phs_vector', 'avg_delta_phs', 'std_delta_phs',
         'gap_phs_vector','avg_gap_phs', 'std_gap_phs']] = results.apply(_row_stats_phs, axis=1)


# --------------------------------------------------------------------- EVF comparison
# Same construction, against the exact value function (EVF) policy. Defined for any
# family that actually has a stored EVF row; NaN/empty otherwise (EVF is only tractable
# for some family sizes).
def _row_stats_evf(r):
    empty = pd.Series({
        'delta_optimal_vector': np.array([]),
        'avg_delta_optimal': np.nan, 'std_delta_optimal': np.nan,
        'avg_gap_optimal': np.nan, 'std_gap_optimal': np.nan,
    })
    fam = _family_of(r)
    evf_rows = fam[fam['policy'] == 'exact_value_function']
    if evf_rows.empty:
        return empty

    evf_cost = -_as_reward_array(evf_rows['all_rewards'].iloc[0])
    cost_pi = -_as_reward_array(r['all_rewards'])
    delta = cost_pi - evf_cost
    avg_evf = evf_cost.mean()
    avg_delta, std_delta = delta.mean(), delta.std(ddof=1)
    return pd.Series({
        'delta_optimal_vector': delta,
        'avg_delta_optimal': avg_delta,
        'std_delta_optimal': std_delta,
        'avg_gap_optimal': avg_delta / avg_evf * 100.0,
        'std_gap_optimal': std_delta / avg_evf * 100.0,
    })

results[['delta_optimal_vector', 'avg_delta_optimal', 'std_delta_optimal',
         'avg_gap_optimal', 'std_gap_optimal']] = results.apply(_row_stats_evf, axis=1)

# quick sanity print (PHS family_gap should be ~0)
_phs_gap = results.loc[results['policy'] == 'perfect_hindsight', 'avg_gap_phs']
print("Consolidation done.", f"results: {len(results)} rows (policy x family).")
if not _phs_gap.empty:
    print(f"  PHS family_gap (should be ~0): {_phs_gap.abs().max():.6f}")

Consolidation done. results: 414 rows (policy x family).
  PHS family_gap (should be ~0): 0.000000


In [2]:
'''
#smaller for export
results_export = results.copy()
results_export = results_export.drop(columns=['all_rewards', 'all_distance_ranks_n_vectors', 'gap_phs_vector', 'delta_phs_vector', 'delta_optimal_vector', 'all_times_n_vectors', 'all_distance_ranks_single_vector', 'all_times_single_vector'])
results_export.to_csv('results/full_tables/full_results_consolidated.csv', index=False)
'''

"\n#smaller for export\nresults_export = results.copy()\nresults_export = results_export.drop(columns=['all_rewards', 'all_distance_ranks_n_vectors', 'gap_phs_vector', 'delta_phs_vector', 'delta_optimal_vector', 'all_times_n_vectors', 'all_distance_ranks_single_vector', 'all_times_single_vector'])\nresults_export.to_csv('results/full_tables/full_results_consolidated.csv', index=False)\n"

In [ ]:
# =====================================================================================
# TABLE(S): per-family sum-ratio gap to PHS + bottom Average/Min/Max rows
# -------------------------------------------------------------------------------------
# Reads ONLY from `results` (family_gap / family_gap_disp), built by the consolidation
# cell. Emits two LaTeX tables:
#   (1) family_gap per family (siunitx S-columns, non-rotated), with bottom
#       Average / Min / Max rows;
#   (2) the same data, rotated/resized, with each cell also showing family_gap_disp,
#       plus the same bottom Average / Min / Max rows.
#
# family_gap is a per-family aggregate, not an average of per-episode ratios:
#       family_gap = (sum_e cost_pi,e - sum_e cost_phs,e) / sum_e cost_phs,e * 100
# i.e. the % gap between the policy's total realised cost over the family's episodes
# and the PHS's total cost over the same episodes. The value in parentheses,
# family_gap_disp, is not a confidence interval - it is the standard deviation of the
# paired per-episode difference between the policy's cost and the PHS's cost within
# that family, scaled by the family's mean PHS cost:
#       family_gap_disp = std_e(cost_pi,e - cost_phs,e) / mean_e(cost_phs,e) * 100
# The bottom "Average" row is the mean of the 32 per-family family_gap values (equal
# weight per family); its paired value is the standard deviation of those 32 values
# (how much the gap varies across families, not within one). Min/Max are the
# smallest/largest per-family family_gap for that policy across all 32 families,
# paired with that family's own family_gap_disp.
# =====================================================================================

# column order + display labels (PHS excluded from the comparison columns)
col_policies = ['imitation_learning', 'myopic', 'genetic_programming',
                'linear_value_function_approximation', 'deep_q_networks',
                'point_estimate_lookahead', 'distributional_estimate_lookahead',
                'linear_programming_heuristic',
                'parameterized_lookahead_approximation', 'proximal_policy_optimization']

_header = r"""    \multicolumn{3}{c|}{\textbf{Instance description}}
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{PPO}\\
"""

_header_plain = r"""    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{myo}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{PPO}\\
"""


def _cell_gap(w, c, d, policy):
    row = results[(results.num_warehouses == w) & (results.num_customers == c) &
                  (results.capacity_distribution == d) & (results.policy == policy)]
    return float(row['avg_gap_phs'].iloc[0]), float(row['std_gap_phs'].iloc[0])


def _avg_gap(policy):
    sub = results[results.policy == policy]
    return float(sub['avg_gap_phs'].mean()), float(sub['avg_gap_phs'].std(ddof=1))


def _minmax_gap(policy, kind):
    sub = results[results.policy == policy]
    idx = sub['avg_gap_phs'].idxmin() if kind == 'min' else sub['avg_gap_phs'].idxmax()
    row = sub.loc[idx]
    return float(row['avg_gap_phs']), float(row['std_gap_phs'])


def _family_rows(with_ci):
    rows = ""
    for num_customers in num_customers_options:
        rows += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, cap in enumerate(capacity_distribution_options):
                if i == 0:
                    rows += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {cap.capitalize()} & "
                else:
                    rows += f"& & {cap.capitalize()} & "
                vals = {p: _cell_gap(num_warehouses, num_customers, cap, p) for p in col_policies}
                best = min(vals, key=lambda p: vals[p][0])
                cells = []
                for p in col_policies:
                    m, h = vals[p]
                    s = f"{m:.2f} ({h:.2f})" if with_ci else f"{m:.2f}"
                    cells.append(f"\\textbf{{{s}}}" if p == best else s)
                rows += " & ".join(cells) + " \\\\\n"
        if num_customers != num_customers_options[-1]:
            rows += "        \\hline\n"
    return rows


def _summary_row(label, values, with_ci):
    best = min(values, key=lambda p: values[p][0])
    cells = []
    for p in col_policies:
        m, h = values[p]
        s = f"{m:.2f} ({h:.2f})" if with_ci else f"{m:.2f}"
        cells.append(f"\\textbf{{{s}}}" if p == best else s)
    return f"\\multicolumn{{3}}{{c|}}{{{label}}} & " + " & ".join(cells) + " \\\\\n"


def _build_gap_table_plain():
    avg = {p: _avg_gap(p) for p in col_policies}
    mn = {p: _minmax_gap(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[H]
    \small
    \centering
    \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the \gls{phs}'s total cost, aggregated over each family's episodes. For each instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
""" + _header_plain + r"""        \toprule
""" + _family_rows(with_ci=False) \
    + "        \\midrule\n" \
        + _summary_row("Average", avg, with_ci=False) \
        + _summary_row("Min", mn, with_ci=False) \
        + _summary_row("Max", mx, with_ci=False) \
        + r"""    \bottomrule
    \end{tabular}
\end{table}
"""
    return t


def _build_gap_table_ci():
    t = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.94\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the \gls{phs}'s total cost, aggregated over each family's episodes, with a paired dispersion measure. For each instance family, every policy is evaluated on the same demand realizations. Each cell reports the average gap to the \gls{phs}, and the value in parentheses is the standard deviation of the paired per-episode difference between the policy's cost and the \gls{phs}'s cost within that family, scaled by the family's mean \gls{phs} cost. The best policy in each row is in bold.}
        \label{tab:policy_comparison_performance_ci}
        \resizebox{0.96\textheight}{!}{%
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|ccc}
    \toprule
""" + _header + r"""        \toprule
""" + _family_rows(with_ci=True) \
        + r"""        \bottomrule

        \end{tabular}%
        }
        \end{minipage}%
    }
\end{table}
"""
    return t


print("% ===== MAIN per-family sum-ratio gap table (Average/Min/Max) =====")
print(_build_gap_table_plain())
print("\n% ===== per-family sum-ratio gap table WITH dispersion (Average/Min/Max) =====")
print(_build_gap_table_ci())

% ===== MAIN per-family sum-ratio gap table (Average/Min/Max) =====

\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the \gls{phs}'s total cost, aggregated over each family's episodes. For each instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap. The best policy in each row is in bold.}
    \label{tab:policy_comparison_performance}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    

In [ ]:
# =====================================================================================
# TABLE: per-family sum-ratio gap to EVF (exact value function)
# -------------------------------------------------------------------------------------
# Same construction and full factorial layout as the PHS gap table above (all 32
# families: num_customers_options x num_warehouses_options x capacity_distribution_options
# as rows), but benchmarked against the exact value function (EVF) policy instead of PHS,
# using the `avg_gap_optimal` / `std_gap_optimal` columns added in the consolidation cell.
# EVF is only tractable for some family sizes, so only families with a stored EVF row get
# populated cells; families without one show "--" instead of being dropped from the table.
# EVF itself is excluded from the comparison columns (its gap to itself is 0 by
# construction), the same treatment PHS gets in the table above. Reuses col_policies,
# _header_plain and _summary_row from the cell above.
# =====================================================================================

_results_optimal = results[results['avg_gap_optimal'].notna()]


def _cell_gap_optimal(w, c, d, policy):
    row = _results_optimal[(_results_optimal.num_warehouses == w) &
                        (_results_optimal.num_customers == c) &
                        (_results_optimal.capacity_distribution == d) &
                        (_results_optimal.policy == policy)]
    if row.empty:
        return None
    return float(row['avg_gap_optimal'].iloc[0]), float(row['std_gap_optimal'].iloc[0])


def _avg_gap_optimal(policy):
    sub = _results_optimal[_results_optimal.policy == policy]
    if sub.empty:
        return float('nan'), float('nan')
    return float(sub['avg_gap_optimal'].mean()), float(sub['avg_gap_optimal'].std(ddof=1))


def _minmax_gap_optimal(policy, kind):
    sub = _results_optimal[_results_optimal.policy == policy]
    if sub.empty:
        return float('nan'), float('nan')
    idx = sub['avg_gap_optimal'].idxmin() if kind == 'min' else sub['avg_gap_optimal'].idxmax()
    row = sub.loc[idx]
    return float(row['avg_gap_optimal']), float(row['std_gap_optimal'])


def _family_rows_optimal():
    rows = ""
    for num_customers in num_customers_options:
        rows += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, cap in enumerate(capacity_distribution_options):
                if i == 0:
                    rows += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {cap.capitalize()} & "
                else:
                    rows += f"& & {cap.capitalize()} & "
                vals = {p: _cell_gap_optimal(num_warehouses, num_customers, cap, p) for p in col_policies}
                cells = []
                for p in col_policies:
                    v = vals[p]
                    s = "" if v is None else f"{v[0]:.2f}"
                    cells.append(s)
                rows += " & ".join(cells) + " \\\\\n"
        if num_customers != num_customers_options[-1]:
            rows += "        \\hline\n"
    return rows


def _build_gap_table_optimal_plain():
    avg = {p: _avg_gap_optimal(p) for p in col_policies}
    mn = {p: _minmax_gap_optimal(p, 'min') for p in col_policies}
    mx = {p: _minmax_gap_optimal(p, 'max') for p in col_policies}
    t = r"""
\begin{table}[H]
    \small
    \centering
     \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the total cost of the optimal policy (solved by backward dynamic programming), aggregated over each family's episodes. Only families with a computed optimal policy are populated; the remaining cells are left blank. For each populated instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap, computed only over the populated families.}
    \label{tab:policy_comparison_performance_evf}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
""" + _header_plain + r"""        \toprule
""" + _family_rows_optimal() \
    + "        \\midrule\n" \
        + _summary_row("Average", avg, with_ci=False) \
        + _summary_row("Min", mn, with_ci=False) \
        + _summary_row("Max", mx, with_ci=False) \
        + r"""    \bottomrule
    \end{tabular}
\end{table}
"""
    return t


print("% ===== OPT per-family sum-ratio gap table (Average/Min/Max) =====")
print(_build_gap_table_optimal_plain())

% ===== OPT per-family sum-ratio gap table (Average/Min/Max) =====

\begin{table}[H]
    \scriptsize
    \centering
     \caption{Policy comparison across all families of instances: percentage gap between each policy's total realized cost and the total cost of the optimal policy (solved by backward dynamic programming), aggregated over each family's episodes. Only families with a computed optimal policy are populated; the remaining cells are left blank. For each populated instance family, every policy is evaluated on the same demand realizations. The bottom summary rows report, for each policy, the average of its per-family gaps (equal weight per family), and the minimum and maximum per-family gap, computed only over the populated families.}
    \label{tab:policy_comparison_performance_evf}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2

In [6]:

policies = ['imitation_learning', 'myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison across all families of instances: mean online runtime per order, in milliseconds. Each cell reports the mean runtime over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean runtimes (equal weight per family), and the minimum and maximum per-family mean runtime. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
    \label{tab:policy_comparison_time}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lvfa}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the percentage above perfect hindsight for each policy
            times = []
            short_times = []
            for policy in policies:
                time = df_filtered.loc[df_filtered['policy'] == policy, 'mean_time'].values[0]
                time = time / 1_000
                if policy not in ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']:
                    time = time / 1_000  # Convert to milliseconds
                else:
                    short_times.append(time)
                times.append(time)

            # Find the min time in times, knowing that short_times contains the relevant ones
            min_time = min(short_times)

            # Format the percentages, highlighting the minimum in bold
            formatted_times = [
                f"\\textbf{{{time:.2f}}}" if time == min_time else f"{time:.2f}"
                for time in times
            ]
            
            latex_table += " & ".join(formatted_times) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\bottomrule\n"

# Add average, min, and max rows
all_min_time = []
all_mean_time = []
all_max_time = []   
for policy in policies:
    all_min_time.append(results.loc[results['policy'] == policy, 'mean_time'].min())
    all_mean_time.append(results.loc[results['policy'] == policy, 'mean_time'].mean())
    all_max_time.append(results.loc[results['policy'] == policy, 'mean_time'].max())

short_time_policies = ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']

def convert_time(value, policy):
    value = value / 1_000
    if policy not in short_time_policies:
        value = value / 1_000  # Convert to milliseconds
    return value

converted_mean_time = [convert_time(v, p) for v, p in zip(all_mean_time, policies)]
converted_min_time = [convert_time(v, p) for v, p in zip(all_min_time, policies)]
converted_max_time = [convert_time(v, p) for v, p in zip(all_max_time, policies)]

best_mean_short = min(v for v, p in zip(converted_mean_time, policies) if p in short_time_policies)
best_mean_long = min(v for v, p in zip(converted_mean_time, policies) if p not in short_time_policies)
best_min_short = min(v for v, p in zip(converted_min_time, policies) if p in short_time_policies)
best_min_long = min(v for v, p in zip(converted_min_time, policies) if p not in short_time_policies)
best_max_short = min(v for v, p in zip(converted_max_time, policies) if p in short_time_policies)
best_max_long = min(v for v, p in zip(converted_max_time, policies) if p not in short_time_policies)

latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy, mean_value in zip(policies, converted_mean_time):
    latex_table += f"& \\textbf{{{mean_value:.2f}}}" if mean_value == best_mean_short else f"& {mean_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy, min_value in zip(policies, converted_min_time):
    latex_table += f"& \\textbf{{{min_value:.2f}}}" if min_value == best_min_short else f"& {min_value:.2f} "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy, max_value in zip(policies, converted_max_time):
    latex_table += f"& \\textbf{{{max_value:.2f}}}" if max_value == best_max_short else f"& {max_value:.2f} "
latex_table += r"\\ \bottomrule"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)


\begin{table}[H]
    \scriptsize
    \centering
    \caption{Policy comparison across all families of instances: mean online runtime per order, in milliseconds. Each cell reports the mean runtime over that family's episodes. The bottom summary rows report, for each policy, the average of its per-family mean runtimes (equal weight per family), and the minimum and maximum per-family mean runtime. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
    \label{tab:policy_comparison_time}
    \begin{tabular}{ccc|S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]|S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above

<>:89: SyntaxWarning: invalid escape sequence '\m'
<>:93: SyntaxWarning: invalid escape sequence '\m'
<>:97: SyntaxWarning: invalid escape sequence '\m'
<>:89: SyntaxWarning: invalid escape sequence '\m'
<>:93: SyntaxWarning: invalid escape sequence '\m'
<>:97: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_42402/1766844055.py:89: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Average} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_42402/1766844055.py:93: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Min} \n"
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_42402/1766844055.py:97: SyntaxWarning: invalid escape sequence '\m'
  latex_table += "\multicolumn{3}{c|}{Max} \n"


In [7]:


policies = ['imitation_learning', 'myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}


latex_table = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.93\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: mean online runtime per order, in milliseconds, with standard deviation. Each cell reports the mean runtime over that family's episodes, with the standard deviation in parentheses. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
        \label{tab:policy_comparison_time_std}
        \resizebox{0.96\textheight}{!}{%
        %\small
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
        \toprule
        \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lvfa}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "
            
            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) & 
                (results["num_warehouses"] == num_warehouses) & 
                (results["capacity_distribution"] == capacity_distribution)
            ]
            
            # Extract the mean (and std) runtime for each policy
            times = []
            stds = []
            short_times = []
            for policy in policies:
                time = df_filtered.loc[df_filtered['policy'] == policy, 'mean_time'].values[0]
                std = df_filtered.loc[df_filtered['policy'] == policy, 'std_time'].values[0]
                if policy in ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']:
                    time = time / 1_000  # Convert to microseconds
                    std = std / 1_000
                    short_times.append(time)
                else:
                    time = time / 1_000_000  # Convert to milliseconds
                    std = std / 1_000_000

                times.append(time)
                stds.append(std)

            # Find the min time in times, knowing that short_times contains the relevant ones
            min_time = min(short_times)

            # Format the times, highlighting the minimum in bold; std shown in parentheses after the mean
            formatted_times = [
                f"\\textbf{{{time:.2f}}} ({std:.2f})" if time == min_time else f"{time:.2f} ({std:.2f})"
                for time, std in zip(times, stds)
            ]
            
            latex_table += " & ".join(formatted_times) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\bottomrule\n"

# Add average, min, and max rows (with standard deviation alongside each)
all_min_time = []
all_mean_time = []
all_max_time = []
all_avg_std_time = []  # between-family std of the per-family means (paired with Average)
all_min_std_time = []  # within-family std of the family attaining the min mean (paired with Min)
all_max_std_time = []  # within-family std of the family attaining the max mean (paired with Max)
for policy in policies:
    policy_rows = results.loc[results['policy'] == policy]
    all_min_time.append(policy_rows['mean_time'].min())
    all_mean_time.append(policy_rows['mean_time'].mean())
    all_max_time.append(policy_rows['mean_time'].max())
    all_avg_std_time.append(policy_rows['mean_time'].std())
    all_min_std_time.append(policy_rows.loc[policy_rows['mean_time'].idxmin(), 'std_time'])
    all_max_std_time.append(policy_rows.loc[policy_rows['mean_time'].idxmax(), 'std_time'])

short_time_policies = ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']

def convert_time(value, policy):
    if policy in short_time_policies:
        return value / 1_000  # Convert to microseconds
    return value / 1_000_000  # Convert to milliseconds

'''
converted_mean_time = [convert_time(v, p) for v, p in zip(all_mean_time, policies)]
converted_min_time = [convert_time(v, p) for v, p in zip(all_min_time, policies)]
converted_max_time = [convert_time(v, p) for v, p in zip(all_max_time, policies)]
converted_avg_std_time = [convert_time(v, p) for v, p in zip(all_avg_std_time, policies)]
converted_min_std_time = [convert_time(v, p) for v, p in zip(all_min_std_time, policies)]
converted_max_std_time = [convert_time(v, p) for v, p in zip(all_max_std_time, policies)]

best_mean_short = min(v for v, p in zip(converted_mean_time, policies) if p in short_time_policies)
best_mean_long = min(v for v, p in zip(converted_mean_time, policies) if p not in short_time_policies)
best_min_short = min(v for v, p in zip(converted_min_time, policies) if p in short_time_policies)
best_min_long = min(v for v, p in zip(converted_min_time, policies) if p not in short_time_policies)
best_max_short = min(v for v, p in zip(converted_max_time, policies) if p in short_time_policies)
best_max_long = min(v for v, p in zip(converted_max_time, policies) if p not in short_time_policies)

latex_table += "\multicolumn{3}{c|}{Average} \n"
for policy, mean_value, std_value in zip(policies, converted_mean_time, converted_avg_std_time):
    latex_table += f"& \\textbf{{{mean_value:.2f}}} ({std_value:.2f})" if mean_value == best_mean_short else f"& {mean_value:.2f} ({std_value:.2f}) "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Min} \n"
for policy, min_value, std_value in zip(policies, converted_min_time, converted_min_std_time):
    latex_table += f"& \\textbf{{{min_value:.2f}}} ({std_value:.2f})" if min_value == best_min_short else f"& {min_value:.2f} ({std_value:.2f}) "
latex_table += r"\\"
latex_table += "\multicolumn{3}{c|}{Max} \n"
for policy, max_value, std_value in zip(policies, converted_max_time, converted_max_std_time):
    latex_table += f"& \\textbf{{{max_value:.2f}}} ({std_value:.2f})" if max_value == best_max_short else f"& {max_value:.2f} ({std_value:.2f}) "
'''


# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
            }%
        \end{minipage}
        }
    \end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.93\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: mean online runtime per order, in milliseconds, with standard deviation. Each cell reports the mean runtime over that family's episodes, with the standard deviation in parentheses. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row is in bold.}
        \label{tab:policy_comparison_time_std}
        \resizebox{0.96\textheight}{!}{%
        %\small
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
        \toprule
        \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{

<>:95: SyntaxWarning: invalid escape sequence '\m'
<>:95: SyntaxWarning: invalid escape sequence '\m'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_42402/1851532553.py:95: SyntaxWarning: invalid escape sequence '\m'
  '''


In [9]:
policies = ['imitation_learning', 'myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'point_estimate_lookahead', 'distributional_estimate_lookahead', 'linear_programming_heuristic','parameterized_lookahead_approximation', 'proximal_policy_optimization']
policies_dict = {'myopic': 'MYO', 'imitation_learning': 'IL', 'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL', 'parameterized_lookahead_approximation': 'CFA-DLA', 'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE', 'proximal_policy_optimization': 'PPO'}

short_time_policies = ['myopic', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'imitation_learning', 'proximal_policy_optimization']


def _time_robust_stats(raw_times, policy):
    """Per-episode raw runtimes -> (median, p95, worst) in the same units as the mean/std table."""
    times_array = np.asarray(raw_times, dtype=float)
    if policy in short_time_policies:
        times_array = times_array / 1_000  # -> microseconds
    else:
        times_array = times_array / 1_000_000  # -> milliseconds
    return (
        float(np.median(times_array)),
        float(np.percentile(times_array, 80)),
        float(np.percentile(times_array, 90)),
        float(np.percentile(times_array, 95)),
        float(np.percentile(times_array, 99)),
        float(times_array.max()),
    )


latex_table = r"""
\begin{table}[!b]
    \centering
    \rotatebox{90}{%
        \begin{minipage}{0.93\textheight}
        \centering
        \caption{\scriptsize Policy comparison across all families of instances: online runtime per order, in milliseconds, robust to outliers. Each cell reports the median runtime over that family's episodes, with the 95th percentile runtime in parentheses. Columns marked with * correspond to very low runtime values and are multiplied by $10^3$; the values in these columns can therefore be read directly as microseconds. The fastest policy in each row (by median) is in bold.}
        \label{tab:policy_comparison_time_robust}
        \resizebox{0.96\textheight}{!}{%
        %\small
        \setlength{\tabcolsep}{8pt}
        \begin{tabular}{ccc|c|cc|cc|cc|cccc}
        \toprule
        \multicolumn{3}{c|}{\textbf{Instance description}} %& \multicolumn{8}{c|}{\textbf{Cost (\% above perfect hindsight)}} \\
    & \multicolumn{1}{c|}{\textbf{\gls{pfa}}} & \multicolumn{2}{c|}{\textbf{\gls{cfa}}} & \multicolumn{2}{c|}{\textbf{\gls{vfa}}} & \multicolumn{2}{c|}{\textbf{\gls{dla}}} & \multicolumn{3}{c}{\textbf{Hybrid}}   \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}\textsuperscript{*}} & \textbf{\gls{myo}\textsuperscript{*}} & \textbf{\gls{gp}\textsuperscript{*}} & \textbf{\gls{lvfa}\textsuperscript{*}} & \textbf{\gls{dqn}\textsuperscript{*}} & \textbf{\gls{pel}} & \textbf{\gls{del}} & \textbf{\gls{lph}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}\textsuperscript{*}}\\
        \toprule
"""

# Loop through the customer and warehouse options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = results[
                (results["num_customers"] == num_customers) &
                (results["num_warehouses"] == num_warehouses) &
                (results["capacity_distribution"] == capacity_distribution)
            ]



            # Extract the median / p80 / p95 / p99 / worst runtime for each policy, from the raw per-episode times
            medians, p80s, p90s, p95s, p99s, worsts, short_medians = [], [], [], [], [], [], []
            for policy in policies:
                raw_times = df_filtered.loc[df_filtered['policy'] == policy, 'all_times_single_vector'].values[0]
                median, p80, p90, p95, p99, worst = _time_robust_stats(raw_times, policy)
                raw_times_arr = np.asarray(raw_times, dtype=float)
                worst3_idx = np.argsort(raw_times_arr)[-3:][::-1]
                print(f"[check] customers={num_customers} warehouses={num_warehouses} capacity={capacity_distribution} policy={policy} worst3_idx={worst3_idx.tolist()} worst3_vals={raw_times_arr[worst3_idx].tolist()}")
                medians.append(median)
                p80s.append(p80)
                p90s.append(p90)
                p95s.append(p95)
                p99s.append(p99)
                worsts.append(worst)
                if policy in short_time_policies:
                    short_medians.append(median)

            # Find the min median in short_medians, knowing that's the relevant comparison group
            min_median = min(short_medians)

            # Format the cells, highlighting the minimum median in bold; p95 and worst shown in parentheses
            formatted_cells = [
                f"\\textbf{{{median:.2f}}} ({p95:.2f})" if median == min_median
                else f"{median:.2f} ({p95:.2f})"
                for median, p80, p90, p95, p99, worst in zip(medians, p80s, p90s, p95s, p99s, worsts)
            ]
            

            latex_table += " & ".join(formatted_cells) + r" \\" + "\n"
        #latex_table += f"        \\cline{{2-{len(policies)+3}}}\n"
    latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
            }%
        \end{minipage}
        }
    \end{table}
"""

# Print the LaTeX table string
print(latex_table)


[check] customers=50 warehouses=2 capacity=uniform policy=imitation_learning worst3_idx=[1524, 1674, 1680] worst3_vals=[2197980.880737, 2106189.727783, 738620.758057]
[check] customers=50 warehouses=2 capacity=uniform policy=myopic worst3_idx=[0, 552, 993] worst3_vals=[4053.115845, 4053.115845, 3099.441528]
[check] customers=50 warehouses=2 capacity=uniform policy=genetic_programming worst3_idx=[413, 8973, 2522] worst3_vals=[64134.597778, 15020.370483, 13828.277588]
[check] customers=50 warehouses=2 capacity=uniform policy=linear_value_function_approximation worst3_idx=[1600, 0, 4976] worst3_vals=[13828.277588, 12874.603271, 12636.184692]
[check] customers=50 warehouses=2 capacity=uniform policy=deep_q_networks worst3_idx=[0, 9265, 326] worst3_vals=[174760.818481, 102281.570435, 98228.45459]
[check] customers=50 warehouses=2 capacity=uniform policy=point_estimate_lookahead worst3_idx=[9768, 3061, 3069] worst3_vals=[937401515960.6934, 232175776958.46558, 132488012.313843]
[check] custom

In [10]:
# =====================================================================================
# DATA PREP for the mixed-effects model + Tukey post-hoc comparison below
# -------------------------------------------------------------------------------------
# Builds the long-format `data` the next cell expects: one row per (family, instance,
# policy), columns ['family_id', 'instance_id', 'policy_id', 'performance'].
#
# performance = that policy's per-episode delta to PHS, normalised by the FAMILY's mean
# PHS cost (not that episode's own PHS cost) - i.e. delta_phs_vector / avg_phs_family *
# 100. This must match avg_gap_phs's normalisation (see the comparison-matrix cell
# above for why episode-own-PHS normalisation would NOT reconcile with the tables), so
# that mixedlm's fixed-effect estimates for C(policy_id) reproduce the same policy
# gaps shown in the family tables.
#
# PHS and EVF are excluded - they aren't competing online policies.
# =====================================================================================
import pandas as pd
import numpy as np

matrix_policies = ['imitation_learning', 'myopic', 'genetic_programming',
                   'linear_value_function_approximation', 'deep_q_networks',
                   'point_estimate_lookahead', 'distributional_estimate_lookahead',
                   'linear_programming_heuristic',
                   'parameterized_lookahead_approximation', 'proximal_policy_optimization']

fam_keys = list(results.groupby(['num_warehouses', 'num_customers', 'capacity_distribution']).groups.keys())

# Family's mean PHS cost, cached once per family (same quantity avg_gap_phs divides by).
avg_phs_by_family = {}
for key in fam_keys:
    fam_phs_rows = results[(results['num_warehouses'] == key[0]) &
                            (results['num_customers'] == key[1]) &
                            (results['capacity_distribution'] == key[2]) &
                            (results['policy'] == 'perfect_hindsight')]
    avg_phs_by_family[key] = _as_reward_array(fam_phs_rows['all_rewards'].iloc[0]).mean() * -1

    '''
    avg_phs_by_family[key]  = results[(results['num_warehouses'] == key[0]) &
                                (results['num_customers'] == key[1]) &
                                (results['capacity_distribution'] == key[2]) &
                                (results['policy'] == 'perfect_hindsight')]['mean_reward'].iloc[0]
    '''
rows = []
for _, r in results[results['policy'].isin(matrix_policies)].iterrows():
    key = (r['num_warehouses'], r['num_customers'], r['capacity_distribution'])
    family_id = f"w{key[0]}_c{key[1]}_{key[2]}"
    #perf = np.asarray(r['delta_phs_vector']) / avg_phs_by_family[key] * 100.0
    #use the reward vector simply
    perf = np.asarray(r['all_rewards']) * -1
    for instance_id, p in enumerate(perf):
        rows.append({
            'family_id': family_id,
            'instance_id': instance_id,
            'policy_id': r['policy'],
            'performance': p,
        })

data = pd.DataFrame(rows)

#Order by policy_id, family_id, instance_id for consistency with the tables above
data = data.sort_values(by=['policy_id', 'family_id', 'instance_id']).reset_index(drop=True)

#Make the order random but dont delete rows (ascending in family_order and instance for same policies but not for other policies

n_fam = data['family_id'].nunique()
n_pol = data['policy_id'].nunique()
inst_per_fam_pol = data.groupby(['family_id', 'policy_id']).size().unique()
print(f"data: {len(data)} rows "
      f"({n_fam} families x {n_pol} policies x {int(list(inst_per_fam_pol)[0])} simulation episodes per family-policy)")

data.to_csv("results/full_tables/data_for_stats.csv", index=False)

data: 64000 rows (32 families x 10 policies x 200 simulation episodes per family-policy)


In [3]:
'''
import scikit_posthocs as sp

#https://scikit-posthocs.readthedocs.io/en/latest/generated/scikit_posthocs.posthoc_wilcoxon.html

# 2. Use the wide-ready Wilcoxon function with Holm-Bonferroni adjustment
# This runs the paired Wilcoxon tests across columns and applies the 'h' correction
wilcoxon_results = sp.posthoc_wilcoxon(
    data, 
    val_col='performance',
    group_col='policy_id',
    p_adjust='holm'
)

print("\n--- 11x11 Wilcoxon Paired P-Value Matrix (Family-Controlled) ---")
#print(wilcoxon_results)
'''


'\nimport scikit_posthocs as sp\n\n#https://scikit-posthocs.readthedocs.io/en/latest/generated/scikit_posthocs.posthoc_wilcoxon.html\n\n# 2. Use the wide-ready Wilcoxon function with Holm-Bonferroni adjustment\n# This runs the paired Wilcoxon tests across columns and applies the \'h\' correction\nwilcoxon_results = sp.posthoc_wilcoxon(\n    data, \n    val_col=\'performance\',\n    group_col=\'policy_id\',\n    p_adjust=\'holm\'\n)\n\nprint("\n--- 11x11 Wilcoxon Paired P-Value Matrix (Family-Controlled) ---")\n#print(wilcoxon_results)\n'

In [4]:
# =====================================================================================
# TABLE: paired Wilcoxon signed-rank pairwise comparison matrix (LaTeX)
# -------------------------------------------------------------------------------------
# Formats `wilcoxon_results` (the Holm-Bonferroni-adjusted p-value matrix from the
# paired-Wilcoxon cell above) into the same visual style as the other comparison
# matrices in this notebook: median of the paired per-instance differences (row
# policy - column policy, in percentage points, computed from `data`) stacked over
# its Holm-adjusted p-value, with significance stars. Reuses `matrix_policies`
# (fixed display order) from the data-prep cell above.
#
# The Wilcoxon signed-rank test evaluates whether the MEDIAN of the paired
# differences is zero, so the effect size shown here is the median of each pair's
# own paired differences (a - b per matched (family, instance) row via `wide_data`),
# not the difference of each policy's marginal median - those are not the same
# quantity in general. This keeps the reported effect size consistent with what the
# test actually tests (mean_diff would not be).
#
# Like the Nemenyi matrix, this one is nonparametric (Wilcoxon signed-rank on the
# paired per-instance differences) and naturally pairs on the matched (family,
# instance) row via `wide_data`, so no separate blocking machinery is needed - but
# unlike Nemenyi it directly tests each pair's own paired differences rather than
# ranks across all 11 policies at once, and multiplicity is corrected here with a
# plain Holm-Bonferroni step-down (statsmodels' multipletests) instead of Nemenyi's
# studentized-range correction.
#   negative => row policy has a smaller gap to PHS => row policy is BETTER.
# =====================================================================================
'''
import numpy as np

matrix_labels = {'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
                  'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
                  'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
                  'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
                  'parameterized_lookahead_approximation': 'PLA', 'proximal_policy_optimization': 'PPO'}
policy_order = matrix_policies  # fixed display order, from the data-prep cell above
labels = [matrix_labels[p] for p in policy_order]

p_matrix_wilcoxon = wilcoxon_results.reindex(index=policy_order, columns=policy_order)

# Pivot to one row per matched (family, instance), one column per policy, so that
# wide_data[a] - wide_data[b] gives the same paired differences the Wilcoxon test
# above is run on.
wide_data = data.pivot(index=['family_id', 'instance_id'], columns='policy_id', values='performance')

n_pol = len(policy_order)
median_diff = np.full((n_pol, n_pol), np.nan)
for ia, a in enumerate(policy_order):
    for ib, b in enumerate(policy_order):
        if ia == ib:
            continue
        median_diff[ia, ib] = (wide_data[a] - wide_data[b]).median()


def _stars(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.01:
        return "^{**}"
    if p < 0.05:
        return "^{*}"
    return ""


col_fmt = "l|" + "r" * n_pol

wt = r"""
\begin{table}[h!]
    \centering
    \scriptsize
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{4pt}
    \label{tab:pairwise_paired_matrix_combined}
    \caption{Nonparametric pairwise comparison of policies' performance gap to \gls{phs} (median of the paired per-instance differences, row policy $-$ column policy, in percentage points), with Holm-Bonferroni-adjusted $p$-values from a paired Wilcoxon signed-rank test on the matched per-instance differences below in parentheses; pairing is on the matched (family, instance) row. Negative values mean the row policy has a smaller gap to the \gls{phs}, and therefore better performance. Significance stars: $^{*}\,p<0.05$, $^{**}\,p<0.01$.}
    \begin{tabular}{""" + col_fmt + r"""}
    \toprule
    \textbf{Row $-$ Col} & """ + " & ".join(f"\\textbf{{{x}}}" for x in labels) + r""" \\
    \midrule
"""
for ia, a in enumerate(policy_order):
    cells = [f"\\textbf{{{labels[ia]}}}"]
    for ib, b in enumerate(policy_order):
        if ia == ib:
            cells.append("")
        else:
            p_val = float(p_matrix_wilcoxon.loc[a, b])
            top = f"${median_diff[ia, ib]:.2f}{_stars(p_val)}$"
            bot = f"({p_val:.3f})"
            cells.append(f"\\shortstack{{{top}\\\\{bot}}}")
    wt += "    " + " & ".join(cells) + r" \\" + "\n"
wt += r"""    \bottomrule
    \end{tabular}
\end{table}
"""
print("% ===== paired Wilcoxon signed-rank pairwise comparison matrix =====")
print(wt)
'''

<>:27: SyntaxWarning: invalid escape sequence '\c'
<>:27: SyntaxWarning: invalid escape sequence '\c'
/var/folders/s2/6ttmr5ds0dd97ztz21jjzx540000gn/T/ipykernel_61765/3720430434.py:27: SyntaxWarning: invalid escape sequence '\c'
  '''


'\nimport numpy as np\n\nmatrix_labels = {\'myopic\': \'MYO\', \'imitation_learning\': \'IL\', \'genetic_programming\': \'GP\',\n                  \'point_estimate_lookahead\': \'PEL\', \'distributional_estimate_lookahead\': \'DEL\',\n                  \'linear_value_function_approximation\': \'LVFA\', \'deep_q_networks\': \'DQN\',\n                  \'linear_programming_heuristic\': \'LPH\', \'linear_programming_exact\': \'LPE\',\n                  \'parameterized_lookahead_approximation\': \'PLA\', \'proximal_policy_optimization\': \'PPO\'}\npolicy_order = matrix_policies  # fixed display order, from the data-prep cell above\nlabels = [matrix_labels[p] for p in policy_order]\n\np_matrix_wilcoxon = wilcoxon_results.reindex(index=policy_order, columns=policy_order)\n\n# Pivot to one row per matched (family, instance), one column per policy, so that\n# wide_data[a] - wide_data[b] gives the same paired differences the Wilcoxon test\n# above is run on.\nwide_data = data.pivot(index=[\'fam

In [11]:
# =====================================================================================
# DATA: per-family paired t-test posthoc (Holm-corrected) + aggregate win/draw matrix
# -------------------------------------------------------------------------------------
# For each of the 32 families, runs a paired t-test (scipy.stats.ttest_rel) on the 200
# matched per-episode `performance` values for every pair of policies (55 pairs among
# the 11 matrix_policies), Holm-corrected across those 55 comparisons within the
# family (statsmodels' multipletests, same correction style as the Wilcoxon table
# above). Unlike scikit_posthocs.posthoc_ttest (independent-samples only), this test
# is PAIRED on the matched (family, instance) row, via `wide_data` from the cell above.
#
# CONFIDENCE_LEVEL is the single knob to change the significance threshold used below
# (alpha = 1 - CONFIDENCE_LEVEL).
#
# print_tables=True additionally prints, for every family, its own 11x11 Holm-adjusted
# p-value matrix (stored in `ttest_posthoc_tables`); left False by default since that's
# 32 tables of limited standalone interest - the aggregate matrix built here is what
# gets exported to LaTeX in the next cell.
#
# ttest_wins[i, j] = number of families (out of 32) in which policy i's gap to PHS is
# significantly smaller than policy j's (i beats j); ttest_draws[i, j] = number of
# families with no significant difference between i and j. By construction,
# ttest_wins[i, j] + ttest_wins[j, i] + ttest_draws[i, j] == 32 for every pair.
# =====================================================================================
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import ttest_rel
#https://docs.scipy.org/doc/scipy-1.15.2/reference/generated/scipy.stats.ttest_rel.html

from statsmodels.stats.multitest import multipletests
#https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html

CONFIDENCE_LEVEL = 0.95
print_tables = False

alpha = 1 - CONFIDENCE_LEVEL
policy_order = matrix_policies
n_pol = len(policy_order)
pairs = list(combinations(range(n_pol), 2))
families = sorted(data['family_id'].unique())
wide_data = data.pivot(index=['family_id', 'instance_id'], columns='policy_id', values='performance')

ttest_wins = np.zeros((n_pol, n_pol), dtype=int)
ttest_draws = np.zeros((n_pol, n_pol), dtype=int)
ttest_posthoc_tables = {}

for family_id in families:
    fam_wide = wide_data.loc[family_id]

    pvals, diffs = [], []
    for ia, ib in pairs:
        a, b = policy_order[ia], policy_order[ib]
        print(fam_wide[a].values)
        _, p = ttest_rel(fam_wide[a].values, fam_wide[b].values)
        pvals.append(p)
        diffs.append(float(np.mean(fam_wide[a].values - fam_wide[b].values)))

    reject, p_adj, _, _ = multipletests(pvals, alpha=alpha, method='holm')

    if print_tables:
        pmat = pd.DataFrame(np.nan, index=policy_order, columns=policy_order)
        for (ia, ib), p in zip(pairs, p_adj):
            pmat.iloc[ia, ib] = pmat.iloc[ib, ia] = p
        ttest_posthoc_tables[family_id] = pmat
        print(f"\n--- paired t-test posthoc (Holm), family {family_id} ---")
        print(pmat.round(4))

    for (ia, ib), p, diff, rej in zip(pairs, p_adj, diffs, reject):
        if rej:
            if diff < 0:
                ttest_wins[ia, ib] += 1
            else:
                ttest_wins[ib, ia] += 1
        else:
            ttest_draws[ia, ib] += 1
            ttest_draws[ib, ia] += 1

# sanity check: row-wins + col-wins + draws == number of families, for every pair
for ia, ib in pairs:
    assert ttest_wins[ia, ib] + ttest_wins[ib, ia] + ttest_draws[ia, ib] == len(families)

print(f"Paired t-test posthoc done: {len(families)} families x {len(pairs)} pairwise comparisons, "
      f"alpha={alpha:.2f} (Holm-corrected within each family).")


[6342.68 6818.86 6326.1  6863.16 6890.34 6398.9  6866.69 6365.35 7309.7
 6745.83 6664.75 6465.62 6785.22 6929.9  7394.01 6525.28 6432.53 7415.48
 6261.72 6949.24 6577.99 6998.55 7099.6  6921.19 7223.45 7076.33 6606.47
 6627.84 6779.06 6801.04 7357.87 6699.55 7275.77 7869.14 7205.21 7170.24
 6757.13 6850.95 6729.83 7215.33 7104.13 6540.46 7103.08 6458.75 7205.53
 6803.12 6467.68 7163.42 6847.   6265.1  6706.77 6790.76 6378.54 7079.17
 7343.15 6289.51 6255.86 6954.21 6842.5  6794.69 7157.1  6791.18 7361.41
 6648.64 6782.62 6394.81 6567.98 7329.69 7149.37 7359.   7163.75 6615.34
 6726.72 7208.87 6557.02 7065.11 6454.53 6837.38 6337.7  7036.14 6578.22
 7190.79 7180.01 6734.89 6640.57 6285.66 7676.82 6606.84 7328.48 6546.25
 7398.17 6822.51 7294.26 7124.42 6668.29 6768.13 7031.68 6526.65 7001.49
 6296.96 7229.6  6425.44 6776.09 6997.45 6541.44 7404.36 6924.69 6932.96
 7059.76 6552.09 6759.09 7031.19 6658.31 6801.06 6491.37 6501.47 7115.
 6714.05 6616.83 7126.46 6257.4  7041.63 7028.01 7332.

In [12]:
wide_data

policy_id                   deep_q_networks  \
family_id      instance_id                    
w2_c100_uneven 0                    6430.59   
               1                    6801.28   
               2                    6461.60   
               3                    6967.39   
               4                    6934.14   
...                                     ...   
w5_c50_uniform 195                  2278.01   
               196                  2073.85   
               197                  2334.41   
               198                  2543.74   
               199                  1966.47   

policy_id                   distributional_estimate_lookahead  \
family_id      instance_id                                      
w2_c100_uneven 0                                      6334.52   
               1                                      6791.74   
               2                                      6326.10   
               3                                      6863.16   
               4                                      6890.34   
...                                                       ...   
w5_c50_uniform 195                                    2183.19   
               196                                    1957.78   
               197                                    2223.51   
               198                                    2542.40   
               199                                    2022.10   

policy_id                   genetic_programming  imitation_learning  \
family_id      instance_id                                            
w2_c100_uneven 0                        6432.15             6342.68   
               1                        6808.22             6818.86   
               2                        6347.50             6326.10   
               3                        6863.16             6863.16   
               4                        6984.78             6890.34   
...                                         ...                 ...   
w5_c50_uniform 195                      2178.66             2126.15   
               196                      2060.17             2048.27   
               197                      2336.36             2299.80   
               198                      2563.85             2516.50   
               199                      1990.07             1906.40   

policy_id                   linear_programming_heuristic  \
family_id      instance_id                                 
w2_c100_uneven 0                                 6342.68   
               1                                 6818.86   
               2                                 6326.10   
               3                                 6863.16   
               4                                 6890.34   
...                                                  ...   
w5_c50_uniform 195                               2191.68   
               196                               1957.43   
               197                               2299.71   
               198                               2506.47   
               199                               1956.40   

policy_id                   linear_value_function_approximation   myopic  \
family_id      instance_id                                                 
w2_c100_uneven 0                                        6469.21  7125.57   
               1                                        6846.41  7597.19   
               2                                        6485.88  7366.62   
               3                                        6928.39  7669.08   
               4                                        7461.52  7828.18   
...                                                         ...      ...   
w5_c50_uniform 195                                      2349.84  2294.51   
               196                                      2084.41  2238.52   
               197                                      2424.58  2381.59   
 

In [14]:
# =====================================================================================
# TABLE: paired t-test win/loss/draw summary matrix across families (LaTeX)
# -------------------------------------------------------------------------------------
# Formats `ttest_wins` / `ttest_draws` (built in the cell above) into an 11x11 matrix:
# each off-diagonal cell shows a / b / c, where a = number of families (out of 32) in
# which the row policy has a significantly smaller gap to PHS than the column policy,
# b = the reverse, and c = number of families with no significant difference - a + b +
# c always sums to 32. Reuses `policy_order` / `matrix_labels` from the cells above.
# Two extra rows are appended below the matrix: a "Total" row that aggregates the
# a/b/c counts of each column policy against all other policies (out of 32 * 10 = 320
# comparisons), and a "Net (W-L)" row with just the difference between the column
# policy's total wins and total losses from that same aggregation.
# =====================================================================================
matrix_labels = {'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
                  'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
                  'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
                  'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
                  'parameterized_lookahead_approximation': 'PLA', 'proximal_policy_optimization': 'PPO'}
labels = [matrix_labels[p] for p in policy_order]
n_families = len(families)
col_fmt = "l|" + "c" * n_pol

tt = r"""
\begin{table}[h!]
    \centering
    \scriptsize
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{4pt}
    \caption{Pairwise comparison of policies' performance across the 32 families of instances. For each pair of policies and each family, a paired $t$-test is conducted on the 200 paired episode-level costs within that family, with $p$-values Holm-corrected across the 55 pairwise comparisons within the family. Each cell reports, out of the 32 families, the number of families in which the row policy has a significantly smaller cost than the column policy, the number in which the column policy has a significantly smaller cost, and the number in which no significant difference is found (row wins / column wins / draws, at $\alpha=0.05$), so that the three values in each cell sum to 32. The \textbf{Net (W$-$L)} row reports, for each column policy, the difference between its total wins and total losses across all 320 pairwise family comparisons (9 opposing policies $\times$ 32 families). This value would only reach $+288$ or $-288$ in the absence of any draws; in practice, draws reduce the maximum attainable magnitude, as reflected, for example, in the \gls{myo} policy's value of $-278$ despite losing every non-drawn comparison.}
    \label{tab:pairwise_ttest_family_summary}
    \begin{tabular}{l|ccccccccccc}
    \toprule
    \textbf{Row $\backslash$ Col} & """ + " & ".join(f"\\textbf{{{x}}}" for x in labels) + r""" \\
    \midrule
"""
for ia, a in enumerate(policy_order):
    cells = [f"\\textbf{{{labels[ia]}}}"]
    for ib, b in enumerate(policy_order):
        if ia == ib:
            cells.append("")
        else:
            cells.append(f"{ttest_wins[ia, ib]}/{ttest_wins[ib, ia]}/{ttest_draws[ia, ib]}")
    tt += "    " + " & ".join(cells) + r" \\" + "\n"

# summary row: for each column policy, aggregate a/b/c (wins / losses / draws) against all other policies
col_summary = []
for j in range(n_pol):
    losses_j = sum(ttest_wins[i, j] for i in range(n_pol) if i != j)
    wins_j = sum(ttest_wins[j, i] for i in range(n_pol) if i != j)
    draws_j = sum(ttest_draws[i, j] for i in range(n_pol) if i != j)
    col_summary.append((wins_j, losses_j, draws_j))

tt += "    \\midrule\n"
#total_cells = [r"\textbf{Total}"] + [f"{losses}/{wins}/{draws}" for wins, losses, draws in col_summary]
#tt += "    " + " & ".join(total_cells) + r" \\" + "\n"
net_cells = [r"\textbf{Net (W$-$L)}"] + [f"{wins - losses:+d}" for wins, losses, draws in col_summary]
tt += "    " + " & ".join(net_cells) + r" \\" + "\n"

tt += r"""    \bottomrule
    \end{tabular}
\end{table}
"""
print("% ===== paired t-test win/loss/draw summary matrix =====")
print(tt)


% ===== paired t-test win/loss/draw summary matrix =====

\begin{table}[h!]
    \centering
    \scriptsize
    \renewcommand{\arraystretch}{1.35}
    \setlength{\tabcolsep}{4pt}
    \caption{Pairwise comparison of policies' performance across the 32 families of instances. For each pair of policies and each family, a paired $t$-test is conducted on the 200 paired episode-level costs within that family, with $p$-values Holm-corrected across the 55 pairwise comparisons within the family. Each cell reports, out of the 32 families, the number of families in which the row policy has a significantly smaller cost than the column policy, the number in which the column policy has a significantly smaller cost, and the number in which no significant difference is found (row wins / column wins / draws, at $\alpha=0.05$), so that the three values in each cell sum to 32. The \textbf{Net (W$-$L)} row reports, for each column policy, the difference between its total wins and total losses across all 320

In [2]:
import pandas as pd

num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ['uniform', 'uneven']

# Read the theta.csv file
theta_df = pd.read_csv('training/linear_value_function_approximation_training/theta.csv')

# Define the LaTeX table structure
latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{lvfa} final weights ($\theta$) across all families of instances.}
    \label{tab:lvfa_theta_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta_A$} & \textbf{$\theta_B$} & \textbf{$\theta_C$} & \textbf{$\theta_D$} & \textbf{$\theta_E$} \\
        \midrule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = theta_df[
                (theta_df["num_customers"] == num_customers) &
                (theta_df["num_warehouses"] == num_warehouses) &
                (theta_df["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the theta values - one coefficient per facility, so pad with
            # blanks up to the 5-column max (num_warehouses_options tops out at 5)
            MAX_THETA_COLS = 5
            if not df_filtered.empty:
                theta = df_filtered.iloc[0]['theta']
                theta_values = [f"{round(float(t), 2)}" for t in theta.split(',')]
            else:
                theta_values = []
            theta_cells = theta_values + [""] * (MAX_THETA_COLS - len(theta_values))
            latex_table += " & ".join(theta_cells) + " \\\\\n"
    latex_table += "        \\hline\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{lvfa} final weights ($\theta$) across all families of instances.}
    \label{tab:lvfa_theta_all}
    \begin{tabular}{ccc|S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]S[table-format=4.2]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta_A$} & \textbf{$\theta_B$} & \textbf{$\theta_C$} & \textbf{$\theta_D$} & \textbf{$\theta_E$} \\
        \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 57.01 & 56.6 &  &  &  \\
& & Uneven & 72.92 & 29.8 &  &  &  \\
& \multirow{2}{*}{3} & Uniform & 55.81 & 31.21 & 52.75 &  &  \\
& & Uneven & 73.99 & 24.82 & 27.22 &  &  \\
& \multirow{2}{*}{4} & Uniform & 42.56 & 43.05 & 37.39 & 37.54 &  \\
& & Uneven & 63.06 & 54.87 & 18.26 & -5.84 &  \\
& \multirow{2}{*}{5} & Uniform & 36.17 & 36.1 & 36.25 & 36.34 & 46.94 \\
& & Uneven & 59.15 & 45.83 & 26.49 & 12.98 & 29.75 \\
        \hli

In [ ]:
# PLA (parameterized lookahead approximation) sensitivity to theta, as a table instead
# of the plot above: per instance, the mean reward at each tested theta plus the
# selected (best) theta. Best = highest mean_reward (reward is negative distance, so
# less negative is better - see Eq. 9/23 as in the plot cell above).
import pandas as pd

param_options = [1.0, 1.02, 1.04, 1.06, 1.08, 1.10]

pla_df = pd.read_csv('training/parameterized_lookahead_approximation_training/parameterized_lookahead_approximation_all_params.csv')

latex_table = r"""
\begin{table}[H]
    \scriptsize
    \centering
    \caption{\gls{pla} mean cost over 50 simulation episodes per tested $\theta$, and selected $\theta$ across all families of instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean cost per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Filter the DataFrame for the current configuration
            df_filtered = pla_df[
                (pla_df["num_customers"] == num_customers) &
                (pla_df["num_warehouses"] == num_warehouses) &
                (pla_df["capacity_distribution"] == capacity_distribution)
            ]

            # Extract the mean reward for each tested theta
            rewards = []
            for param in param_options:
                reward = df_filtered.loc[df_filtered['param'] == param, 'mean_reward'].values[0]
                rewards.append(reward)

            # Best theta = highest mean reward (least negative)
            best_reward = max(rewards)
            best_param = param_options[rewards.index(best_reward)]

            formatted_rewards = [
                f"\\textbf{{{-reward:.2f}}}" if reward == best_reward else f"{-reward:.2f}"
                for reward in rewards
            ]

            latex_table += " & ".join(formatted_rewards) + f" & {best_param:.2f} " + r" \\" + "\n"
    latex_table += "        \\hline\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[H]
    \scriptsize
    \centering
    \caption{\gls{pla} mean cost per tested $\theta$ and selected $\theta$ across all families of instances.}
    \label{tab:pla_theta_sensitivity}
    \begin{tabular}{ccc|S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]S[table-format=5.2]|S[table-format=1.2]}
    \toprule
    \multicolumn{3}{c|}{\textbf{Instance description}} & \multicolumn{6}{c|}{\textbf{Mean reward per tested $\theta$}} & \\
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$\theta=1.00$} & \textbf{$\theta=1.02$} & \textbf{$\theta=1.04$} & \textbf{$\theta=1.06$} & \textbf{$\theta=1.08$} & \textbf{$\theta=1.10$} & \textbf{Selected $\theta$} \\
        \toprule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & \textbf{3267.67} & 3269.50 & 3270.93 & 3277.07 & 3282.80 & 3288.08 & 1.00  \\
& & Uneven & \textbf{3485.22} & 3486.65 & 3486.22 & 3487.86 & 3487.89 & 3486.19 & 1.00  \\
& 

In [13]:
from env import InventoryEnv

# Define the LaTeX table structure - same layout as the LVFA theta table above, but
# each cell is that family's initial capacity for facility A/B/C/D/E (deterministic
# given num_warehouses/num_customers/capacity_distribution, see InventoryEnv.__init__
# in env.py) instead of a learned weight.
latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{Initial facility capacity across all families of instances.}
    \label{tab:facility_initial_capacity_all}
    \begin{tabular}{ccc|S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$A$} & \textbf{$B$} & \textbf{$C$} & \textbf{$D$} & \textbf{$E$} \\
        \midrule
"""

# Loop through the customer, warehouse, and capacity options to fill the table
for num_customers in num_customers_options:
    latex_table += f"        \\multirow{{8}}{{*}}{{{num_customers}}} "
    for num_warehouses in num_warehouses_options:
        for i, capacity_distribution in enumerate(capacity_distribution_options):
            if i == 0:
                latex_table += f"& \\multirow{{2}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
            else:
                latex_table += f"& & {capacity_distribution.capitalize()} & "

            # Initial capacity per facility - one value per warehouse, so pad with
            # blanks up to the 5-column max (num_warehouses_options tops out at 5)
            MAX_FACILITY_COLS = 5
            env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
            capacity_values = [str(v) for v in env.warehouses_initial_capacity]
            capacity_cells = capacity_values + [""] * (MAX_FACILITY_COLS - len(capacity_values))
            latex_table += " & ".join(capacity_cells) + " \\\\\n"
    latex_table += "        \\hline\n"
latex_table += "        \\bottomrule\n"

# Close the LaTeX table structure
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)



\begin{table}[H]
    \centering
    \scriptsize
    \caption{Initial facility capacity across all families of instances.}
    \label{tab:facility_initial_capacity_all}
    \begin{tabular}{ccc|S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]S[table-format=3.0]}
        \toprule
        \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{$A$} & \textbf{$B$} & \textbf{$C$} & \textbf{$D$} & \textbf{$E$} \\
        \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 25 & 25 &  &  &  \\
& & Uneven & 35 & 15 &  &  &  \\
& \multirow{2}{*}{3} & Uniform & 17 & 17 & 16 &  &  \\
& & Uneven & 25 & 15 & 10 &  &  \\
& \multirow{2}{*}{4} & Uniform & 13 & 13 & 12 & 12 &  \\
& & Uneven & 20 & 15 & 10 & 5 &  \\
& \multirow{2}{*}{5} & Uniform & 10 & 10 & 10 & 10 & 10 \\
& & Uneven & 16 & 12 & 10 & 7 & 5 \\
        \hline
        \multirow{8}{*}{100} & \multirow{2}{*}{2} & Uniform & 50 & 50 &  &  &  \\
& & Uneven & 70 & 30 &  &  & 

In [14]:
# Imitation learning (IL) neural network - architecture and training hyperparameters.
# Values are read from policies/auxiliaries/imitation_learning_model.py (architecture,
# shared with inference) and training/train_imitation_learning.ipynb (training loop),
# not from a results CSV, since these are fixed configuration rather than measured
# outputs.
il_hyperparameters = [
    ("Imitation target (expert policy)", "Exact value function (backward induction)"),
    ("Training instances per family", r"5{,}000 ($|\OrderSet| = 50$ fixed)"),
    ("Input features", r"$2|\FacilitySet|$ (normalized distance and remaining capacity per facility)"),
    ("Architecture", r"Dense($2|\FacilitySet| \to 64$) - ReLU - Dense($64 \to 16$) - ReLU - Dense($16 \to |\FacilitySet|$)"),
    ("Output", r"logits over $|\FacilitySet|$ facilities (predicted action = argmax)"),
    ("Loss function", "Cross-entropy"),
    ("Optimizer", "Adam (default learning rate $= 0.001$)"),
    ("Batch size", "64"),
    ("Epochs", "25"),
    ("Train / validation / test split", r"64\% / 16\% / 20\%"),
    ("Random seed (data split)", "42"),
]

latex_table = r"""
\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{il} neural network architecture and training hyperparameters.}
    \label{tab:il_hyperparameters}
    \begin{tabular}{ll}
        \toprule
        \textbf{Parameter} & \textbf{Value} \\
        \midrule
"""

for name, value in il_hyperparameters:
    latex_table += f"        {name} & {value} \\\\\n"

latex_table += r"""        \bottomrule
    \end{tabular}
\end{table}
"""

print(latex_table)



\begin{table}[H]
    \centering
    \scriptsize
    \caption{\gls{il} neural network architecture and training hyperparameters.}
    \label{tab:il_hyperparameters}
    \begin{tabular}{ll}
        \toprule
        \textbf{Parameter} & \textbf{Value} \\
        \midrule
        Imitation target (expert policy) & Exact value function (backward induction) \\
        Training instances per family & 5{,}000 ($|\OrderSet| = 50$ fixed) \\
        Input features & $2|\FacilitySet|$ (normalized distance and remaining capacity per facility) \\
        Architecture & Dense($2|\FacilitySet| \to 64$) - ReLU - Dense($64 \to 16$) - ReLU - Dense($16 \to |\FacilitySet|$) \\
        Output & logits over $|\FacilitySet|$ facilities (predicted action = argmax) \\
        Loss function & Cross-entropy \\
        Optimizer & Adam (default learning rate $= 0.001$) \\
        Batch size & 64 \\
        Epochs & 25 \\
        Train / validation / test split & 64\% / 16\% / 20\% \\
        Random seed (data spl

In [12]:
# =====================================================================================
# TABLE: policy training time (in minutes), per family of instances
# -------------------------------------------------------------------------------------
# Provide a vector of policy names; for each one this reads
# training/{policy}_training/{policy}_training_times.csv (columns: num_warehouses,
# num_customers, capacity_distribution, training_time) and emits a LaTeX table with one
# row per family (same ccc + multirow layout as the other per-family tables above) and
# one column per policy. Unlike the runtime-comparison table above, no
# Average/Min/Max summary rows are added here - just the raw per-family training time.
# A policy missing its CSV, or missing a specific family within it, gets a dash ('--').
# =====================================================================================
import os
import pandas as pd


def generate_training_time_table(
    policies,
    num_warehouses_options=(2, 3, 4, 5),
    num_customers_options=(50, 100, 200, 400),
    capacity_distribution_options=('uniform', 'uneven'),
    policies_dict=None,
    caption=None,
    label='tab:policy_training_time',
):
    """Build a LaTeX table with the training time (in minutes) of each policy in
    `policies`, for each family of instances."""
    default_labels = {
        'myopic': 'MYO', 'imitation_learning': 'IL', 'genetic_programming': 'GP',
        'linear_value_function_approximation': 'LVFA', 'deep_q_networks': 'DQN',
        'point_estimate_lookahead': 'PEL', 'distributional_estimate_lookahead': 'DEL',
        'linear_programming_heuristic': 'LPH', 'linear_programming_exact': 'LPE',
        'parameterized_lookahead_approximation': 'PLA',
        'proximal_policy_optimization': 'PPO', 'exact_value_function': 'OPT',
    }
    labels = {**default_labels, **(policies_dict or {})}

    # Load each policy's training times into {(w, c, d): training_time}
    times_by_policy = {}
    for policy in policies:
        csv_path = f'training/{policy}_training/{policy}_training_times.csv'
        if not os.path.exists(csv_path):
            print(f"[warning] no training-time CSV for '{policy}' at {csv_path}; "
                  f"filling that column with '--'")
            times_by_policy[policy] = {}
            continue
        df = pd.read_csv(csv_path)
        times_by_policy[policy] = {
            (int(r.num_warehouses), int(r.num_customers), r.capacity_distribution): float(r.training_time)
            for r in df.itertuples()
        }

    # Column width (integer digits) per policy, from its own observed max value
    col_format = {}
    for policy in policies:
        values = list(times_by_policy[policy].values())
        max_int_digits = len(str(int(max(values)))) if values else 1
        col_format[policy] = max(max_int_digits, 1)

    header_cols = " & ".join(f"\\textbf{{\\gls{{{labels.get(p, p).lower()}}}}}" for p in policies)
    col_spec = "ccc|" + "".join(f"S[table-format={col_format[p]}.2]" for p in policies)

    if caption is None:
        caption = ("Training time (in minutes) of each policy across all families of "
                   "instances.")

    latex_table = "\\begin{table}[H]\n"
    latex_table += "    \\scriptsize\n    \\centering\n"
    latex_table += f"    \\caption{{{caption}}}\n"
    latex_table += f"    \\label{{{label}}}\n"
    latex_table += f"    \\begin{{tabular}}{{{col_spec}}}\n"
    latex_table += "    \\toprule\n"
    latex_table += (f"    \\textbf{{$|\\OrderSet|$}} & \\textbf{{$|\\FacilitySet|$}} & "
                     f"\\textbf{{Capacity}} & {header_cols} \\\\\n")
    latex_table += "    \\midrule\n"

    rows_per_family = len(num_warehouses_options) * len(capacity_distribution_options)
    for f_idx, num_customers in enumerate(num_customers_options):
        latex_table += f"        \\multirow{{{rows_per_family}}}{{*}}{{{num_customers}}} "
        for num_warehouses in num_warehouses_options:
            for i, capacity_distribution in enumerate(capacity_distribution_options):
                if i == 0:
                    latex_table += f"& \\multirow{{{len(capacity_distribution_options)}}}{{*}}{{{num_warehouses}}} & {capacity_distribution.capitalize()} & "
                else:
                    latex_table += f"& & {capacity_distribution.capitalize()} & "

                cells = []
                for policy in policies:
                    value = times_by_policy[policy].get((num_warehouses, num_customers, capacity_distribution))
                    cells.append(f"{value:.2f}" if value is not None else "")
                latex_table += " & ".join(cells) + r" \\" + "\n"
        if f_idx < len(num_customers_options) - 1:
            latex_table += "        \\hline\n"

    latex_table += "        \\bottomrule\n"
    latex_table += "    \\end{tabular}\n\\end{table}\n"
    return latex_table


# Provide the policies to include (must have a training/{policy}_training/{policy}_training_times.csv)
training_time_policies = ['imitation_learning', 'genetic_programming', 'linear_value_function_approximation', 'deep_q_networks', 'parameterized_lookahead_approximation', 'proximal_policy_optimization', 'exact_value_function']

print(generate_training_time_table(training_time_policies))


\begin{table}[H]
    \scriptsize
    \centering
    \caption{Training time (in minutes) of each policy across all families of instances.}
    \label{tab:policy_training_time}
    \begin{tabular}{ccc|S[table-format=1.2]S[table-format=1.2]S[table-format=1.2]S[table-format=2.2]S[table-format=1.2]S[table-format=2.2]S[table-format=2.2]}
    \toprule
    \textbf{$|\OrderSet|$} & \textbf{$|\FacilitySet|$} & \textbf{Capacity} & \textbf{\gls{il}} & \textbf{\gls{gp}} & \textbf{\gls{lvfa}} & \textbf{\gls{dqn}} & \textbf{\gls{pla}} & \textbf{\gls{ppo}} & \textbf{\gls{opt}} \\
    \midrule
        \multirow{8}{*}{50} & \multirow{2}{*}{2} & Uniform & 0.43 & 0.46 & 0.27 & 7.46 & 0.52 & 12.22 & 0.00 \\
& & Uneven & 0.42 & 0.31 & 0.27 & 7.44 & 0.53 & 12.07 & 0.00 \\
& \multirow{2}{*}{3} & Uniform & 0.44 & 1.30 & 0.32 & 7.54 & 0.74 & 12.20 & 0.00 \\
& & Uneven & 0.44 & 0.39 & 0.31 & 7.53 & 0.75 & 12.25 & 0.00 \\
& \multirow{2}{*}{4} & Uniform & 0.44 & 0.67 & 0.36 & 7.42 & 0.97 & 12.16 & 0.01 \\
& & Unev